# 02 — Label census and integrity audit

Loads both versions, harmonises columns, and produces the descriptive tables.

This notebook answers: *how much actually changed?* — before any model is trained.
Its output is Table 1 of the paper.

Reference values to check against (from the Engelen et al. project erratum):
**9,103** `Attempted` labels, **1,657,069** benign flows in the corrected version.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)

import pandas as pd, numpy as np, json
pd.set_option('display.width', 200)

Mounted at /content/drive


In [2]:
DAYS = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']

# Published per-file row counts for CICIDS2017 (MachineLearningCVE reference).
# Total 2,830,743.
REFERENCE_ROWS = {
    'Monday-WorkingHours': 529_918,
    'Tuesday-WorkingHours': 445_909,
    'Wednesday-workingHours': 692_703,
    'Thursday-WorkingHours-Morning-WebAttacks': 170_366,
    'Thursday-WorkingHours-Afternoon-Infilteration': 288_602,
    'Friday-WorkingHours-Morning': 191_033,
    'Friday-WorkingHours-Afternoon-PortScan': 286_467,
    'Friday-WorkingHours-Afternoon-DDos': 225_745,
}
REFERENCE_TOTAL = sum(REFERENCE_ROWS.values())


def day_of(filename):
    f = filename.lower()
    for d in DAYS:
        if d in f:
            return d
    return None


def shrink(df):
    """float64 -> float32, int64 -> int32 where safe. Halves memory."""
    for c in df.columns:
        t = df[c].dtype
        if t == 'float64':
            df[c] = df[c].astype('float32')
        elif t == 'int64' and df[c].abs().max() < 2_147_000_000:
            df[c] = df[c].astype('int32')
    return df


def load_version(root, version, path_filter=None, exclude=()):
    """Read every CSV under root, tag with day and version, harmonise columns.

    path_filter : only read directories whose path contains this substring.
    exclude     : skip any file whose name contains one of these substrings.

    Both guards matter here. The original mirror ships GeneratedLabelledFlows
    AND MachineLearningCVE; the improved mirror ships CICIDS2017_improved AND
    CSE-CIC-IDS2018_improved, whose filenames also contain weekday names.
    """
    frames = []
    for dirpath, _, files in os.walk(root):
        if path_filter and path_filter.lower() not in dirpath.lower():
            continue
        for fn in sorted(files):
            if not fn.lower().endswith('.csv'):
                continue
            if any(x.lower() in fn.lower() for x in exclude):
                print('  excluded:', fn)
                continue
            d = day_of(fn)
            if d is None:
                print('  skipping (no day in name):', fn)
                continue
            p = os.path.join(dirpath, fn)
            df = pd.read_csv(p, encoding='latin-1', low_memory=False)
            df = H.harmonise(df)
            df = shrink(df)
            df['day'] = d
            df['version'] = version
            frames.append(df)
            print(f'  {version:9s} {d:10s} {len(df):>9,} rows  {fn}')
    assert frames, f'No usable CSVs under {root} (filter={path_filter})'
    return pd.concat(frames, ignore_index=True)


print('ORIGINAL  (TrafficLabelling_ only)')
orig = load_version(C.RAW_ORIGINAL, 'original', path_filter='trafficlabelling')

print('\nIMPROVED  (2017 only — 2018 files excluded)')
impr = load_version(C.RAW_IMPROVED, 'improved',
                    path_filter='cicids2017_improved',
                    exclude=('2018',))

print(f'\noriginal: {len(orig):,} rows, {orig.shape[1]} cols')
print(f'improved: {len(impr):,} rows, {impr.shape[1]} cols')
print(f'\npublished reference for the original: {REFERENCE_TOTAL:,} flows')
print(f'observed minus reference: {len(orig) - REFERENCE_TOTAL:+,}')

if len(impr) > 3_000_000:
    raise RuntimeError('Improved row count is too high — CSE-CIC-IDS2018 files are '
                       'still being read. Check path_filter and exclude.')

ORIGINAL  (TrafficLabelling_ only)
  original  friday       225,745 rows  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  original  friday       286,467 rows  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  original  friday       191,033 rows  Friday-WorkingHours-Morning.pcap_ISCX.csv
  original  monday       529,918 rows  Monday-WorkingHours.pcap_ISCX.csv
  original  thursday     288,602 rows  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  original  thursday     458,968 rows  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  original  tuesday      445,909 rows  Tuesday-WorkingHours.pcap_ISCX.csv
  original  wednesday    692,703 rows  Wednesday-workingHours.pcap_ISCX.csv

IMPROVED  (2017 only — 2018 files excluded)
  improved  friday       547,557 rows  friday.csv
  improved  monday       371,624 rows  monday.csv
  improved  thursday     362,076 rows  thursday.csv
  improved  tuesday      322,078 rows  tuesday.csv
  improved  wednesday    496,641 rows  w

## Reconciliation against published counts

Seven of the eight original files should match the canonical per-file counts exactly (2,830,743 total). Any file that does not is a data-integrity finding and goes in the paper — it is invisible to anyone using `MachineLearningCVE`.

In [3]:
# Per-file reconciliation against the published counts.
#
# This is a result, not just a check. If a file disagrees with its published
# row count, that discrepancy belongs in the paper.

import re

rows = []
for dirpath, _, files in os.walk(C.RAW_ORIGINAL):
    if 'trafficlabelling' not in dirpath.lower():
        continue
    for fn in sorted(f for f in files if f.lower().endswith('.csv')):
        stem = fn.replace('.pcap_ISCX.csv', '').replace('.csv', '')
        n = sum(1 for _ in open(os.path.join(dirpath, fn),
                                encoding='latin-1', errors='replace')) - 1
        ref = REFERENCE_ROWS.get(stem)
        rows.append({
            'file': stem,
            'observed': n,
            'published': ref if ref is not None else '',
            'delta': (n - ref) if ref is not None else '',
        })

recon = pd.DataFrame(rows)
H.save_table(recon, 'table0_file_reconciliation.csv')
display(recon)

anomalies = recon[(recon['delta'] != '') & (recon['delta'] != 0)]
if len(anomalies):
    print('\nDISCREPANCIES FOUND:')
    for r in anomalies.itertuples():
        print(f'  {r.file}: {r.observed:,} observed vs {r.published:,} published '
              f'({r.delta:+,})')
    print('\nCheck whether any delta equals another file\'s row count exactly —')
    print('that pattern means one file\'s contents were appended to another.')
    obs = dict(zip(recon['file'], recon['observed']))
    for r in anomalies.itertuples():
        for other, cnt in obs.items():
            if other != r.file and cnt == r.delta:
                print(f'  -> {r.file} excess ({r.delta:,}) == {other} total. '
                      f'Likely a concatenation artefact.')

saved /content/drive/MyDrive/research/ids-label-correction/results/table0_file_reconciliation.csv (8, 4)


,file,observed,published,delta
0,Friday-WorkingHours-Afternoon-DDos,225745,225745,0
1,Friday-WorkingHours-Afternoon-PortScan,286467,286467,0
2,Friday-WorkingHours-Morning,191033,191033,0
3,Monday-WorkingHours,529918,529918,0
4,Thursday-WorkingHours-Afternoon-Infilteration,288602,288602,0
5,Thursday-WorkingHours-Morning-WebAttacks,458968,170366,288602
6,Tuesday-WorkingHours,445909,445909,0
7,Wednesday-workingHours,692703,692703,0



DISCREPANCIES FOUND:
  Thursday-WorkingHours-Morning-WebAttacks: 458,968 observed vs 170,366 published (+288,602)

Check whether any delta equals another file's row count exactly —
that pattern means one file's contents were appended to another.
  -> Thursday-WorkingHours-Morning-WebAttacks excess (288,602) == Thursday-WorkingHours-Afternoon-Infilteration total. Likely a concatenation artefact.


In [4]:
# If the reconciliation flagged Thursday-Morning-WebAttacks, look inside it.
# Does the excess carry the Infiltration day's labels, or duplicate flow IDs?

target = None
for dirpath, _, files in os.walk(C.RAW_ORIGINAL):
    if 'trafficlabelling' not in dirpath.lower():
        continue
    for fn in files:
        if 'webattacks' in fn.lower():
            target = os.path.join(dirpath, fn)

if target:
    tw = H.harmonise(pd.read_csv(target, encoding='latin-1', low_memory=False))
    print(target)
    print('\nrows:', len(tw))
    print('\nlabel distribution:')
    print(tw['label'].value_counts())

    if 'flow_id' in tw.columns:
        dup = tw['flow_id'].duplicated().sum()
        print(f'\nduplicated flow ids: {dup:,} ({dup/len(tw):.1%})')

    if 'timestamp' in tw.columns:
        ts = pd.to_datetime(tw['timestamp'], errors='coerce', dayfirst=True)
        print('\ntimestamp range:', ts.min(), '->', ts.max())
        print('(a morning-only capture spanning into the afternoon is the tell)')
else:
    print('WebAttacks file not found.')

/content/drive/MyDrive/research/ids-label-correction/data/raw_original/GeneratedLabelledFlows/TrafficLabelling /Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv

rows: 458968

label distribution:
label
BENIGN                        168186
Web Attack  Brute Force        1507
Web Attack  XSS                 652
Web Attack  Sql Injection        21
Name: count, dtype: int64

duplicated flow ids: 369,101 (80.4%)

timestamp range: 2017-07-06 08:59:00 -> 2017-07-06 12:59:00
(a morning-only capture spanning into the afternoon is the tell)


In [5]:
# Forensics on the 288,602-row excess.
#
# The labelled rows in the WebAttacks file sum to exactly the published 170,366.
# The excess therefore carries NO label. Three questions, answered directly:
#   1. Are the unlabelled rows real flows, or malformed/blank lines?
#   2. Do they match the Infiltration file's flows?
#   3. Where do they sit in the file?

import numpy as np

tw = H.harmonise(pd.read_csv(target, encoding='latin-1', low_memory=False))
null_mask = tw['label'].isna() | (tw['label'].astype(str).str.strip() == '')

print(f'rows in file        : {len(tw):,}')
print(f'rows with a label   : {(~null_mask).sum():,}   (published: 170,366)')
print(f'rows with NO label  : {null_mask.sum():,}')

nulls = tw.loc[null_mask]

# Q1 — are they populated?
probe_cols = [c for c in ['src_ip', 'dst_ip', 'src_port', 'dst_port',
                          'protocol', 'timestamp', 'flow duration']
              if c in tw.columns]
print('\nnon-null rate within the unlabelled block:')
for c in probe_cols:
    print(f'  {c:15s} {nulls[c].notna().mean():.3f}')

# Q2 — position in the file
idx = np.where(null_mask.values)[0]
if len(idx):
    print(f'\nunlabelled rows occupy positions {idx.min():,} to {idx.max():,}')
    contiguous = (idx.max() - idx.min() + 1) == len(idx)
    print('contiguous block:', contiguous)

# Q3 — do they match the Infiltration file?
inf_path = target.replace('Morning-WebAttacks', 'Afternoon-Infilteration')
if os.path.exists(inf_path):
    inf = H.harmonise(pd.read_csv(inf_path, encoding='latin-1', low_memory=False))
    print(f'\nInfiltration file rows: {len(inf):,}')

    key = [c for c in ['src_ip', 'src_port', 'dst_ip', 'dst_port',
                       'protocol', 'timestamp'] if c in tw.columns and c in inf.columns]
    if key:
        def keyset(d):
            return set(map(tuple, d[key].astype(str).values))
        ks_null, ks_inf = keyset(nulls), keyset(inf)
        overlap = len(ks_null & ks_inf)
        print(f'key columns used: {key}')
        print(f'unlabelled keys      : {len(ks_null):,}')
        print(f'infiltration keys    : {len(ks_inf):,}')
        print(f'overlap              : {overlap:,} '
              f'({overlap / max(len(ks_null), 1):.1%} of the unlabelled block)')
        print()
        if overlap / max(len(ks_null), 1) > 0.9:
            print('VERDICT: the unlabelled block IS the Infiltration capture, '
                  'appended without labels.')
        elif overlap / max(len(ks_null), 1) < 0.1:
            print('VERDICT: NOT the Infiltration data. The row-count match is '
                  'coincidental; these are unlabelled flows of separate origin.')
        else:
            print('VERDICT: partial overlap — inspect further before writing this up.')
else:
    print('\nInfiltration file not found for comparison.')

rows in file        : 458,968
rows with a label   : 170,366   (published: 170,366)
rows with NO label  : 288,602

non-null rate within the unlabelled block:
  src_ip          0.000
  dst_ip          0.000
  src_port        0.000
  dst_port        0.000
  protocol        0.000
  timestamp       0.000
  flow duration   0.000

unlabelled rows occupy positions 170,366 to 458,967
contiguous block: True

Infiltration file rows: 288,602
key columns used: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp']
unlabelled keys      : 1
infiltration keys    : 269,212
overlap              : 0 (0.0% of the unlabelled block)

VERDICT: NOT the Infiltration data. The row-count match is coincidental; these are unlabelled flows of separate origin.


In [6]:
# Repair: drop rows with no label, and record what was dropped.
#
# Rows without a label cannot be used for supervised learning under any policy,
# so this is not a judgement call. The COUNT is a reported result.

null_before = int(orig['label'].isna().sum() +
                  (orig['label'].astype(str).str.strip() == '').sum())

repair_log = {
    'original_rows_loaded': int(len(orig)),
    'published_reference': int(REFERENCE_TOTAL),
    'null_label_rows_dropped': null_before,
}

mask = orig['label'].notna() & (orig['label'].astype(str).str.strip() != '')
orig = orig.loc[mask].reset_index(drop=True)

repair_log['original_rows_after_repair'] = int(len(orig))
repair_log['matches_published_reference'] = bool(len(orig) == REFERENCE_TOTAL)

import json
with open(os.path.join(C.RESULTS, 'repair_log.json'), 'w') as f:
    json.dump(repair_log, f, indent=2)

print(json.dumps(repair_log, indent=2))
print()
if repair_log['matches_published_reference']:
    print('Row count now matches the published 2,830,743 exactly.')
else:
    print('Still off by', len(orig) - REFERENCE_TOTAL, '- investigate before continuing.')

{
  "original_rows_loaded": 3119345,
  "published_reference": 2830743,
  "null_label_rows_dropped": 288602,
  "original_rows_after_repair": 2830743,
  "matches_published_reference": true
}

Row count now matches the published 2,830,743 exactly.


In [7]:
# Gate G1 — integrity check against published figures
#
# IMPORTANT: the reference figures below come from the Engelen et al. WTMC 2021
# project erratum. The Kaggle mirror we are using is the Liu et al. CNS 2022
# release, which was regenerated with a later version of the fixed CICFlowMeter.
# The two are NOT expected to agree exactly. Treat a few percent as a version
# difference to document, not a pipeline fault. A large gap still means stop.

benign_impr = int((impr['label'].astype(str).str.strip().str.upper() == 'BENIGN').sum())
attempted_mask = H.is_attempted(impr['label'])
attempted_n = int(attempted_mask.sum())

REF_BENIGN, REF_ATTEMPTED = 1_657_069, 9_103

print(f'benign flows (improved): {benign_impr:>10,}   WTMC2021 reference {REF_BENIGN:>10,}'
      f'   ({(benign_impr - REF_BENIGN) / REF_BENIGN:+.2%})')
print(f'attempted flows:         {attempted_n:>10,}   WTMC2021 reference {REF_ATTEMPTED:>10,}'
      f'   ({(attempted_n - REF_ATTEMPTED) / REF_ATTEMPTED:+.2%})')

print('\nverdict:')
for name, got, ref, tol in [('benign', benign_impr, REF_BENIGN, 0.10),
                            ('attempted', attempted_n, REF_ATTEMPTED, 1.00)]:
    dev = abs(got - ref) / ref
    status = 'PASS' if dev <= tol else 'INVESTIGATE'
    print(f'  {name:10s} deviation {dev:7.2%}  (tolerance {tol:.0%})  -> {status}')

print('\nBoth deviations are expected to be non-zero. Record the observed values;')
print('they become this study\'s own reference, cited alongside the erratum figures.')

with open(os.path.join(C.RESULTS, 'gate_g1.json'), 'w') as f:
    json.dump({'benign_improved': benign_impr,
               'attempted_improved': attempted_n,
               'ref_benign_wtmc2021': REF_BENIGN,
               'ref_attempted_wtmc2021': REF_ATTEMPTED}, f, indent=2)

benign flows (improved):  1,582,566   WTMC2021 reference  1,657,069   (-4.50%)
attempted flows:             11,979   WTMC2021 reference      9,103   (+31.59%)

verdict:
  benign     deviation   4.50%  (tolerance 10%)  -> PASS
  attempted  deviation  31.59%  (tolerance 100%)  -> PASS

Both deviations are expected to be non-zero. Record the observed values;
they become this study's own reference, cited alongside the erratum figures.


In [8]:
# Table 1 — class distribution, both versions
def census(df, version):
    fam = H.coarse_class(df['label'])
    t = fam.value_counts().rename_axis('class').reset_index(name=f'n_{version}')
    t[f'pct_{version}'] = (t[f'n_{version}'] / len(df) * 100).round(3)
    return t

t_orig = census(orig, 'original')
t_impr = census(impr, 'improved')

table1 = t_orig.merge(t_impr, on='class', how='outer').fillna(0)
table1['delta_n'] = table1['n_improved'] - table1['n_original']
table1 = table1.sort_values('n_original', ascending=False)

H.save_table(table1, 'table1_class_census.csv')
table1

saved /content/drive/MyDrive/research/ids-label-correction/results/table1_class_census.csv (9, 6)


,class,n_original,pct_original,n_improved,pct_improved,delta_n
0,BENIGN,2273097,80.300,1582566,75.361,-690531
4,DoS,252661,8.926,177510,8.453,-75151
7,PortScan,158930,5.614,159066,7.575,136
3,DDoS,128027,4.523,95144,4.531,-32883
2,BruteForce,13835,0.489,6972,0.332,-6863
8,WebAttack,2180,0.077,2056,0.098,-124
1,Bot,1966,0.069,4803,0.229,2837
6,Infiltration,36,0.001,71848,3.421,71812
5,Heartbleed,11,0.000,11,0.001,0


In [9]:
# Duplicate audit — protocol section 6 step 5
dup_rows = []
for df, v in [(orig, 'original'), (impr, 'improved')]:
    feats = H.feature_columns(df)
    n = len(df)
    n_dedup = len(df.drop_duplicates(subset=feats + ['label']))
    dup_rows.append({
        'version': v,
        'rows': n,
        'unique': n_dedup,
        'exact_duplicates': n - n_dedup,
        'duplicate_rate_pct': round((n - n_dedup) / n * 100, 3),
    })

dups = pd.DataFrame(dup_rows)
H.save_table(dups, 'table2_duplicates.csv')
dups

saved /content/drive/MyDrive/research/ids-label-correction/results/table2_duplicates.csv (2, 5)


,version,rows,unique,exact_duplicates,duplicate_rate_pct
0,original,2830743,2812414,18329,0.647
1,improved,2099976,2099973,3,0.000


In [10]:
# Per-day comparison — where did the flow counts move?
by_day = (pd.concat([
    orig.groupby('day').size().rename('original'),
    impr.groupby('day').size().rename('improved'),
], axis=1).reindex(DAYS).fillna(0).astype(int))
by_day['delta'] = by_day['improved'] - by_day['original']
by_day['pct_change'] = (by_day['delta'] / by_day['original'] * 100).round(2)
by_day = by_day.reset_index().rename(columns={'index': 'day'})

H.save_table(by_day, 'table3_flows_per_day.csv')
by_day

saved /content/drive/MyDrive/research/ids-label-correction/results/table3_flows_per_day.csv (5, 5)


,day,original,improved,delta,pct_change
0,monday,529918,371624,-158294,-29.87
1,tuesday,445909,322078,-123831,-27.77
2,wednesday,692703,496641,-196062,-28.30
3,thursday,458968,362076,-96892,-21.11
4,friday,703245,547557,-155688,-22.14


In [11]:
# Attempted-label breakdown — the H3 knob
#
# The 'attempted category' column uses a sentinel for non-attempted flows, so
# notna() matches every row. Detect from the label string instead.

att_mask = H.is_attempted(impr['label'])
print(f'flows marked Attempted: {int(att_mask.sum()):,} '
      f'({att_mask.mean():.3%} of the improved dataset)')

att = (impr.loc[att_mask]
       .groupby(['day', 'label']).size()
       .reset_index(name='n')
       .sort_values('n', ascending=False))
H.save_table(att, 'table4_attempted_breakdown.csv')
display(att)

print('\nby family:')
fam = H.coarse_class(impr.loc[att_mask, 'label']).value_counts()
print(fam)

# How much of each attack family is merely attempted? This is the H3 stakes.
all_fam = H.coarse_class(impr['label']).value_counts()
share = (fam / all_fam * 100).dropna().sort_values(ascending=False).round(2)
print('\nattempted share of each family (%):')
print(share)

flows marked Attempted: 11,979 (0.570% of the improved dataset)
saved /content/drive/MyDrive/research/ids-label-correction/results/table4_attempted_breakdown.csv (11, 3)


,day,label,n
0,friday,Botnet - Attempted,4067
9,wednesday,DoS Slowhttptest - Attempted,3368
10,wednesday,DoS Slowloris - Attempted,1847
2,thursday,Web Attack - Brute Force - Attempted,1292
4,thursday,Web Attack - XSS - Attempted,655
8,wednesday,DoS Hulk - Attempted,581
7,wednesday,DoS GoldenEye - Attempted,80
1,thursday,Infiltration - Attempted,45
6,tuesday,SSH-Patator - Attempted,27
5,tuesday,FTP-Patator - Attempted,12



by family:
DoS             5876
Bot             4067
WebAttack       1952
Infiltration      45
BruteForce        39
Name: count, dtype: int64

attempted share of each family (%):
WebAttack       94.94
Bot             84.68
DoS              3.31
BruteForce       0.56
Infiltration     0.06
Name: count, dtype: float64


In [12]:
# Cache the harmonised frames so notebook 03 does not re-parse everything
import pyarrow  # noqa: F401  (parquet engine)

orig.to_parquet(os.path.join(C.INTERIM, 'original.parquet'), index=False)
impr.to_parquet(os.path.join(C.INTERIM, 'improved.parquet'), index=False)
print('cached to', C.INTERIM)

cached to /content/drive/MyDrive/research/ids-label-correction/data/interim


## What to write up from this notebook

Table 1 is already a result. If the class census shows large movement in specific families
(the DoS Hulk and Infiltration classes are the ones prior work flags), say so plainly with
the numbers — that paragraph is the setup for everything that follows.

Next: `03_match.ipynb`.